MiniLM + Random forest

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib
import os

# Load your data
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')
df = df.sample(n=100000, random_state=42)

# Fill missing text with empty string
df['title'] = df['title'].fillna('')

# Select features
text_col = 'title'
tabular_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
target_col = '2_way_label'

# Split into train/val/test (64/16/20)
X = df[[text_col] + tabular_cols]
y = df[target_col]

# 75% Train (750k), 15% Val (150k), 10% Test (100k)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.90, random_state=42, stratify=y_temp)



In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp)

In [ ]:
from sentence_transformers import SentenceTransformer
import time
start_time = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embeddings(text_series):
    start = time.time()
    embeddings = model.encode(
        text_series.tolist(),
        batch_size=256,
        show_progress_bar=True
    )
    print(f"Embedding extraction for {len(text_series)} samples took {((time.time()-start)/60):.2f} minutes")
    return embeddings

# Extract embeddings for each split
train_embeddings = get_embeddings(X_train[text_col])
val_embeddings = get_embeddings(X_val[text_col])
test_embeddings = get_embeddings(X_test[text_col])
end_time = time.time()
print(f"Total training time: {end_time - start_time:.2f} seconds")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Embedding extraction for 64000 samples took 11.83 minutes


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embedding extraction for 16000 samples took 2.62 minutes


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embedding extraction for 20000 samples took 3.27 minutes
⏱️ Total training time: 1080.82 seconds


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[tabular_cols])  # Fit only on train

train_tabular = scaler.transform(X_train[tabular_cols])
val_tabular = scaler.transform(X_val[tabular_cols])
test_tabular = scaler.transform(X_test[tabular_cols])


In [ ]:
import numpy as np

X_train_all = np.hstack([train_embeddings, train_tabular])
X_val_all = np.hstack([val_embeddings, val_tabular])
X_test_all = np.hstack([test_embeddings, test_tabular])


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train_all, y_train)

# Validation performance
y_val_pred = clf.predict(X_val_all)
print("Validation Results:")
print(classification_report(y_val, y_val_pred))
from sklearn.metrics import roc_auc_score

# Get predicted probabilities for the positive class (usually class '1')
probs = clf.predict_proba(X_val_all)[:, 1]

# Calculate AUC
auc_score = roc_auc_score(y_val, probs)
print(f"Validation set AUC: {auc_score:.4f}")

Validation Results:
              precision    recall  f1-score   support

           0       0.93      0.66      0.77      6800
           1       0.80      0.96      0.87      9200

    accuracy                           0.84     16000
   macro avg       0.86      0.81      0.82     16000
weighted avg       0.85      0.84      0.83     16000

Validation set AUC: 0.9210


In [ ]:
y_test_pred = clf.predict(X_test_all)
print("Test Results:")
print(classification_report(y_test, y_test_pred))
from sklearn.metrics import roc_auc_score

# Get predicted probabilities for the positive class (usually class '1')
probs = clf.predict_proba(X_test_all)[:, 1]

# Calculate AUC
auc_score = roc_auc_score(y_test, probs)
print(f"AUC: {auc_score:.4f}")


Test Results:
              precision    recall  f1-score   support

           0       0.93      0.66      0.77      8501
           1       0.79      0.96      0.87     11499

    accuracy                           0.84     20000
   macro avg       0.86      0.81      0.82     20000
weighted avg       0.85      0.84      0.83     20000

AUC: 0.9220


MiniLM + ReLU based neural network

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib
import os
# Load your data
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')

# Fill missing text with empty string
df['title'] = df['title'].fillna('')

# Select features
text_col = 'title'
tabular_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
target_col = '2_way_label'


# Split into train/val/test (64/16/20)
X = df[[text_col] + tabular_cols]
y = df[target_col]

# 75% Train (750k), 15% Val (150k), 10% Test (100k)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.90, random_state=42, stratify=y_temp)

In [ ]:
from sentence_transformers import SentenceTransformer
import time
start_time = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embeddings(text_series):
    start = time.time()
    embeddings = model.encode(
        text_series.tolist(),
        batch_size=256,
        show_progress_bar=True
    )
    print(f"Embedding extraction for {len(text_series)} samples took {((time.time()-start)/60):.2f} minutes")
    return embeddings

# Extract embeddings for each split
train_embeddings = get_embeddings(X_train[text_col])
val_embeddings = get_embeddings(X_val[text_col])
test_embeddings = get_embeddings(X_test[text_col])
end_time = time.time()
print(f"⏱️ Total training time: {end_time - start_time:.2f} seconds")


Batches:   0%|          | 0/293 [00:00<?, ?it/s]

Embedding extraction for 75000 samples took 13.17 minutes


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Embedding extraction for 15000 samples took 2.37 minutes


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Embedding extraction for 10000 samples took 1.67 minutes
⏱️ Total training time: 1033.18 seconds


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[tabular_cols])  # Fit only on train
os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/scaler.pkl")
print("✅ Scaler saved!")

train_tabular = scaler.transform(X_train[tabular_cols])
val_tabular = scaler.transform(X_val[tabular_cols])
test_tabular = scaler.transform(X_test[tabular_cols])


✅ Scaler saved!


In [ ]:
import numpy as np

X_train_all = np.hstack([train_embeddings, train_tabular])
X_val_all = np.hstack([val_embeddings, val_tabular])
X_test_all = np.hstack([test_embeddings, test_tabular])


In [ ]:
# Replace NaNs with 0.0, and Infs with large finite numbers
X_train_all = np.nan_to_num(X_train_all, nan=0.0, posinf=1e5, neginf=-1e5)
X_val_all = np.nan_to_num(X_val_all, nan=0.0, posinf=1e5, neginf=-1e5)
X_test_all = np.nan_to_num(X_test_all, nan=0.0, posinf=1e5, neginf=-1e5)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
# Ensure clean targets


input_layer = Input(shape=(X_train_all.shape[1],))
x = Dense(128, activation='relu')(input_layer)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)

relu_model = Model(inputs=input_layer, outputs=output)
relu_model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Train
relu_model.fit(X_train_all, y_train, validation_data=(X_val_all, y_val), epochs=5, batch_size=256)
relu_model.save("fake_news_model.h5")


Epoch 1/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7594 - loss: 0.4935 - val_accuracy: 0.8529 - val_loss: 0.3401
Epoch 2/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8556 - loss: 0.3328 - val_accuracy: 0.8707 - val_loss: 0.3060
Epoch 3/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8725 - loss: 0.3039 - val_accuracy: 0.8771 - val_loss: 0.2914
Epoch 4/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8859 - loss: 0.2798 - val_accuracy: 0.8826 - val_loss: 0.2834
Epoch 5/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8898 - loss: 0.2652 - val_accuracy: 0.8848 - val_loss: 0.2749


In [ ]:
# === Validation Set Evaluation ===
val_probs = relu_model.predict(X_val_all, batch_size=256)
val_preds = (val_probs > 0.5).astype(int)

print("\nValidation Classification Report:")
print(classification_report(y_val, val_preds))

if np.isnan(val_probs).any():
    print("⚠️ Warning: NaNs found in validation probabilities. Skipping AUC.")
else:
    val_auc = roc_auc_score(y_val, val_probs)
    print(f"Validation AUC: {val_auc:.4f}")

# === Test Set Evaluation ===
test_probs = relu_model.predict(X_test_all, batch_size=256)
test_preds = (test_probs > 0.5).astype(int)

print("\nTest Classification Report:")
print(classification_report(y_test, test_preds))

if np.isnan(test_probs).any():
    print("⚠️ Warning: NaNs found in test probabilities. Skipping AUC.")
else:
    test_auc = roc_auc_score(y_test, test_probs)
    print(f"Test AUC: {test_auc:.4f}")


59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.83      0.86      6376
           1       0.88      0.93      0.90      8624

    accuracy                           0.88     15000
   macro avg       0.89      0.88      0.88     15000
weighted avg       0.89      0.88      0.88     15000

Validation AUC: 0.9504
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

Test Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.84      0.87      4250
           1       0.89      0.93      0.91      5750

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

Test AUC: 0.9534


MiniLM + NN with subregion

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Load data
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')

# Fill missing values
df['title'] = df['title'].fillna('')
df['sub_region'] = df['sub_region'].fillna('unknown')  # fill missing regions

# Encode sub_region (One-Hot)
region_dummies = pd.get_dummies(df['sub_region'], prefix='region')
df = pd.concat([df, region_dummies], axis=1)

# Define columns
text_col = 'title'
tabular_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
region_cols = region_dummies.columns.tolist()
all_tabular = tabular_cols + region_cols
target_col = '2_way_label'

# Features and targets
X = df[[text_col] + all_tabular]
y = df[target_col]

# Split train/val/test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.90, random_state=42, stratify=y_temp)

# ================= Embedding Extraction =================
from sentence_transformers import SentenceTransformer
import time
start_time = time.time()
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embeddings(text_series):
    start = time.time()
    embeddings = model.encode(
        text_series.tolist(),
        batch_size=256,
        show_progress_bar=True
    )
    print(f"Embedding extraction for {len(text_series)} samples took {((time.time()-start)/60):.2f} minutes")
    return embeddings

train_embeddings = get_embeddings(X_train[text_col])
val_embeddings = get_embeddings(X_val[text_col])
test_embeddings = get_embeddings(X_test[text_col])
end_time = time.time()
print(f"Total training time: {end_time - start_time:.2f} seconds")

# ================= Tabular Data Processing =================
scaler = StandardScaler()
scaler.fit(X_train[all_tabular])  # Fit on train only
# os.makedirs("model", exist_ok=True)
# joblib.dump(scaler, "model/scaler.pkl")
# print("Scaler saved!")

train_tabular = scaler.transform(X_train[all_tabular])
val_tabular = scaler.transform(X_val[all_tabular])
test_tabular = scaler.transform(X_test[all_tabular])

# Combine embeddings + tabular
X_train_all = np.hstack([train_embeddings, train_tabular])
X_val_all = np.hstack([val_embeddings, val_tabular])
X_test_all = np.hstack([test_embeddings, test_tabular])

# Clean NaN/Inf
X_train_all = np.nan_to_num(X_train_all, nan=0.0, posinf=1e5, neginf=-1e5)
X_val_all = np.nan_to_num(X_val_all, nan=0.0, posinf=1e5, neginf=-1e5)
X_test_all = np.nan_to_num(X_test_all, nan=0.0, posinf=1e5, neginf=-1e5)

# ================= Model =================
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, roc_auc_score

input_layer = Input(shape=(X_train_all.shape[1],))
x = Dense(128, activation='relu')(input_layer)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)

relu_model = Model(inputs=input_layer, outputs=output)
relu_model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Train
relu_model.fit(X_train_all, y_train, validation_data=(X_val_all, y_val), epochs=5, batch_size=256)
# relu_model.save("fake_news_model.h5")

# ================= Validation Evaluation =================
val_probs = relu_model.predict(X_val_all, batch_size=256)
val_preds = (val_probs > 0.5).astype(int)

print("\nValidation Classification Report:")
print(classification_report(y_val, val_preds))

if np.isnan(val_probs).any():
    print("Warning: NaNs found in validation probabilities. Skipping AUC.")
else:
    val_auc = roc_auc_score(y_val, val_probs)
    print(f"Validation AUC: {val_auc:.4f}")

# ================= Test Evaluation =================
test_probs = relu_model.predict(X_test_all, batch_size=256)
test_preds = (test_probs > 0.5).astype(int)

print("\nTest Classification Report:")
print(classification_report(y_test, test_preds))

if np.isnan(test_probs).any():
    print("Warning: NaNs found in test probabilities. Skipping AUC.")
else:
    test_auc = roc_auc_score(y_test, test_probs)
    print(f"Test AUC: {test_auc:.4f}")


Batches:   0%|          | 0/293 [00:00<?, ?it/s]

Embedding extraction for 75000 samples took 13.44 minutes


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Embedding extraction for 15000 samples took 2.39 minutes


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Embedding extraction for 10000 samples took 1.70 minutes
Total training time: 1051.98 seconds
Epoch 1/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7330 - loss: 0.5250 - val_accuracy: 0.8476 - val_loss: 0.3542
Epoch 2/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8479 - loss: 0.3562 - val_accuracy: 0.8602 - val_loss: 0.3264
Epoch 3/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8630 - loss: 0.3247 - val_accuracy: 0.8697 - val_loss: 0.3083
Epoch 4/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8720 - loss: 0.3057 - val_accuracy: 0.8754 - val_loss: 0.2982
Epoch 5/5
293/293 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8797 - loss: 0.2914 - val_accuracy: 0.8807 - val_loss: 0.2915
59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.83      0.86      6376
           1       0.88      0.92      0.90      8624

    accuracy                     

# Hybrid bert

Hybrid bert without subregion

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')
df = df.dropna(subset=['title'])

# Fill missing numerical values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
df[numeric_cols] = df[numeric_cols].fillna(0)



# Final numeric feature set
all_numeric = numeric_cols

# Extract input and target
X_text = df['title'].values
X_numeric = df[all_numeric].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
X_text_tv, X_text_test, X_num_tv, X_num_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.15, stratify=y, random_state=42
)

X_text_train, X_text_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features (fit on train)
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, labels):
        self.texts = texts
        self.numerics = numerics
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTModel(nn.Module):
    def __init__(self, numeric_input_dim):
        super(HybridBERTModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        combined = torch.cat((cls_output, numerics), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTModel(numeric_input_dim=X_num_train.shape[1]).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== VALIDATION METRICS ==================
def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))

    if np.isnan(all_probs).any():
        print(f"Warning: NaNs found in {dataset_name.lower()} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# ================== EVALUATE ==================
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1 Loss: 307.7189
Epoch 2 Loss: 216.0888
Epoch 3 Loss: 139.5160

Total training time: 1163.64 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.85      0.88      6377
         1.0       0.89      0.94      0.92      8626

    accuracy                           0.90     15003
   macro avg       0.91      0.90      0.90     15003
weighted avg       0.90      0.90      0.90     15003

Validation AUC: 0.9655

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.86      0.89      6375
         1.0       0.90      0.95      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.91      0.90      0.91     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9676


Hybrid bert with subregion

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')
df = df.dropna(subset=['title'])

# Fill missing numerical and region values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')

# One-hot encode sub_region
region_dummies = pd.get_dummies(df['sub_region'], prefix='region')
df = pd.concat([df, region_dummies], axis=1)

# Final numeric feature set
region_cols = region_dummies.columns.tolist()
all_numeric = numeric_cols + region_cols

# Extract input and target
X_text = df['title'].values
X_numeric = df[all_numeric].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
# First: 85% train+val, 15% test
X_text_tv, X_text_test, X_num_tv, X_num_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.15, stratify=y, random_state=42
)

# Then: 82.35% train, 17.65% val
X_text_train, X_text_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features (fit on train)
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, labels):
        self.texts = texts
        self.numerics = numerics
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTModel(nn.Module):
    def __init__(self, numeric_input_dim):
        super(HybridBERTModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        combined = torch.cat((cls_output, numerics), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTModel(numeric_input_dim=X_num_train.shape[1]).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== VALIDATION METRICS ==================
from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))

    if np.isnan(all_probs).any():
        print(f"Warning: NaNs found in {dataset_name.lower()} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Call the evaluation function for both validation and test sets
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1 Loss: 309.5698
Epoch 2 Loss: 216.9313
Epoch 3 Loss: 139.7331

Total training time: 1179.23 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.86      0.89      6377
         1.0       0.90      0.94      0.92      8626

    accuracy                           0.91     15003
   macro avg       0.91      0.90      0.90     15003
weighted avg       0.91      0.91      0.91     15003

Validation AUC: 0.9655

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.87      0.89      6375
         1.0       0.91      0.94      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.91      0.91      0.91     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9660


Region prefix propting

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')
df = df.dropna(subset=['title'])

# Fill missing numerical and region values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')
df['title'] = "region_" + df['sub_region'] + ": " + df['title']

# One-hot encode sub_region
region_dummies = pd.get_dummies(df['sub_region'], prefix='region')
df = pd.concat([df, region_dummies], axis=1)

# Final numeric feature set
region_cols = region_dummies.columns.tolist()
all_numeric = numeric_cols + region_cols

# Extract input and target
X_text = df['title'].values
X_numeric = df[all_numeric].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
# First: 85% train+val, 15% test
X_text_tv, X_text_test, X_num_tv, X_num_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.15, stratify=y, random_state=42
)

# Then: 82.35% train, 17.65% val
X_text_train, X_text_val, X_num_train, X_num_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features (fit on train)
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, labels):
        self.texts = texts
        self.numerics = numerics
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTModel(nn.Module):
    def __init__(self, numeric_input_dim):
        super(HybridBERTModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        combined = torch.cat((cls_output, numerics), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTModel(numeric_input_dim=X_num_train.shape[1]).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== VALIDATION METRICS ==================
from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))

    if np.isnan(all_probs).any():
        print(f"Warning: NaNs found in {dataset_name.lower()} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Call the evaluation function for both validation and test sets
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1 Loss: 313.6965
Epoch 2 Loss: 224.8167
Epoch 3 Loss: 150.5371

Total training time: 1148.28 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.89      0.89      6377
         1.0       0.92      0.91      0.92      8626

    accuracy                           0.91     15003
   macro avg       0.90      0.90      0.90     15003
weighted avg       0.91      0.91      0.91     15003

Validation AUC: 0.9666

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.90      0.89      6375
         1.0       0.92      0.92      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.90      0.91      0.91     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9669


HybridBERT-RE

In [ ]:
# Hybrid BERT + Numeric + Region Embedding
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
df = pd.read_csv('/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/fakeddit_combined_100k_balanced.csv')
df = df.dropna(subset=['title'])

# Fill missing values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')

# Region ID encoding
df['region_id'] = df['sub_region'].astype('category').cat.codes
region2id = dict(enumerate(df['sub_region'].astype('category').cat.categories))
num_regions = len(region2id)

# Final feature sets
all_numeric = numeric_cols
X_text = df['title'].values
X_numeric = df[all_numeric].values
X_region = df['region_id'].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
X_text_tv, X_text_test, X_num_tv, X_num_test, X_reg_tv, X_reg_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, X_region, y, test_size=0.15, stratify=y, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, X_reg_train, X_reg_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, X_reg_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, region_ids, labels):
        self.texts = texts
        self.numerics = numerics
        self.region_ids = region_ids
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'region_id': torch.tensor(self.region_ids[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, X_reg_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, X_reg_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, X_reg_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTRegionEmbeddingModel(nn.Module):
    def __init__(self, numeric_input_dim, num_regions, region_embed_dim=16):
        super(HybridBERTRegionEmbeddingModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.region_embedding = nn.Embedding(num_regions, region_embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim + region_embed_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics, region_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        region_embeds = self.region_embedding(region_ids)
        combined = torch.cat((cls_output, numerics, region_embeds), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTRegionEmbeddingModel(
    numeric_input_dim=X_num_train.shape[1],
    num_regions=num_regions
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        region_ids = batch['region_id'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics, region_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== EVALUATION ==================
def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            region_ids = batch['region_id'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics, region_ids)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))
    if np.isnan(all_probs).any():
        print(f"Warning: NaNs in {dataset_name} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Evaluate
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")


Epoch 1 Loss: 310.3023
Epoch 2 Loss: 217.4016
Epoch 3 Loss: 141.5937

Total training time: 1135.43 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.88      0.89      6377
         1.0       0.92      0.93      0.92      8626

    accuracy                           0.91     15003
   macro avg       0.91      0.91      0.91     15003
weighted avg       0.91      0.91      0.91     15003

Validation AUC: 0.9667

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.89      0.90      6375
         1.0       0.92      0.93      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.91      0.91      0.91     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9671
